# E6: Feature Selection Funnel — L2 (Stability of Risk Relationship Over Time)

**Spec sections**: 6.3, 6.8, D.5  
**Input**: DEV GOOD+BAD; features with `L1_pass = Y`; L1 binning objects (fixed, not refitted).  
**Output**: `iv_s1..iv_s4`, `rank_corr_min`, `head_tail_flip`, `L2_flag`, `L2_flag_reason`, `L2_pass`.

In [0]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, hashlib, shutil, tempfile, json, time, pickle, warnings
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import common

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
BINNING_DIR = os.path.join(ARTIFACTS, "binning", "L1")
os.makedirs(REPORTS, exist_ok=True)

cat = cfg["databricks"]["catalog"]
sch = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L2 = cfg["funnel"]["L2"]
N_SEG          = L2["n_segments"]
SEG_IV_MIN     = L2["seg_iv_min"]
DROP_LOW_GE    = L2["drop_if_low_iv_segments_ge"]
MIN_BIN_CT     = L2["min_bin_count_for_rank"]
RANK_CORR_FLAG = L2["rank_corr_flag"]
IV_RATIO_FLAG  = L2["iv_ratio_flag"]
DROP_HT_FLIP   = L2.get("drop_on_head_tail_flip", True)

target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]
SEED = cfg["random_seed"]

# Funnel config hash check (AGENTS.md: E5–E9 recompute and stop if changed)
funnel_config_hash = hashlib.sha256(
    json.dumps(cfg["funnel"], sort_keys=True).encode()
).hexdigest()

run_summary_path = os.path.join(REPORTS, "run_summaries", "s03_L0.json")
assert os.path.exists(run_summary_path), f"L0 run summary not found: {run_summary_path}"
with open(run_summary_path) as f:
    l0_summary = json.load(f)
l0_funnel_hash = l0_summary["funnel_config_hash"]

tc_path = os.path.join(PROJECT_ROOT, "config", "threshold_changes.csv")
if funnel_config_hash != l0_funnel_hash:
    if os.path.exists(tc_path):
        tc = pd.read_csv(tc_path)
        print(f"\u26a0 Funnel config hash changed but threshold_changes.csv has {len(tc)} row(s) — proceeding.")
        print(tc.to_string())
    else:
        raise ValueError(
            f"Funnel config hash mismatch!\n"
            f"  L0 frozen : {l0_funnel_hash[:12]}\n"
            f"  Current   : {funnel_config_hash[:12]}\n"
            f"Add a row to config/threshold_changes.csv explaining the change."
        )
else:
    print(f"\u2705 Funnel config hash matches L0: {funnel_config_hash[:12]}")

_TMP = tempfile.mkdtemp(prefix="e6_")
_t0 = time.time()

print(f"Config hash       : {common.config_hash(cfg)[:12]}")
print(f"Master table      : {MASTER_TBL}")
print(f"L2 params         : n_seg={N_SEG}, seg_iv_min={SEG_IV_MIN}, "
      f"drop_low_ge={DROP_LOW_GE}, min_bin_ct={MIN_BIN_CT}, "
      f"rank_corr_flag={RANK_CORR_FLAG}, iv_ratio_flag={IV_RATIO_FLAG}")

In [0]:
# ── Load master table and feature log ───────────────────────────────────────────
master = spark.table(MASTER_TBL).toPandas()
dev = master[master["window"] == "DEV"].copy()

# L2 input: DEV GOOD + BAD only
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])].copy()
common.assert_labelled(dev_gb, cfg)
y = (dev_gb[status_col] == "BAD").astype(int)
dev_gb[target_col] = y   # ensure binary for equal_bad_segments
print(f"DEV GOOD+BAD rows : {len(dev_gb):,}  |  Bads: {y.sum()}")

# Feature log
flog = spark.table(FLOG_TBL).toPandas()
print(f"Feature log rows  : {len(flog)}")

# L1 survivors
l1_pass_feats = flog[flog["L1_pass"] == "Y"]["feature"].tolist()
l1_pass_feats = [f for f in l1_pass_feats if f in dev_gb.columns]
print(f"L1 survivors (in data): {len(l1_pass_feats)}")

# Load L1 binning objects and hash them for integrity check
binning_objects = {}
binning_hashes  = {}
for feat in l1_pass_feats:
    pkl_path = os.path.join(BINNING_DIR, f"{feat}.pkl")
    assert os.path.exists(pkl_path), f"Missing L1 binning object: {pkl_path}"
    with open(pkl_path, "rb") as f:
        raw = f.read()
    binning_hashes[feat] = hashlib.sha256(raw).hexdigest()
    binning_objects[feat] = pickle.loads(raw)
print(f"Loaded {len(binning_objects)} L1 binning objects")

In [0]:
# ── Equal-Bad segments (spec 6.3.1, D.5) ──────────────────────────────
seg_map = common.equal_bad_segments(dev_gb, cfg)   # Series: app_month -> S1..S4
dev_gb["segment"] = dev_gb["app_month"].map(seg_map)
assert dev_gb["segment"].notna().all(), "Some months unmapped to segments"

# Report period boundaries
period_labels = sorted(seg_map.unique())
print(f"\nPeriod boundaries:")
for seg in period_labels:
    months = seg_map[seg_map == seg].index.sort_values()
    sub = dev_gb[dev_gb["segment"] == seg]
    n = len(sub)
    bads = y[sub.index].sum()
    br = bads / n if n else 0
    print(f"  {seg}: {months[0]} – {months[-1]}  |  n={n:,}  Bads={bads}  BR={br:.4f}")

# Summary for acceptance
period_bads = dev_gb.groupby("segment")[target_col].sum()
print(f"\nBad counts per period: {dict(period_bads)}")
mean_bads = period_bads.mean()
print(f"Mean Bads={mean_bads:.1f}, range=[{period_bads.min()}, {period_bads.max()}]")

In [0]:
# ── Per-feature period statistics (spec 6.3.2) ──────────────────────

def compute_iv_from_counts(good_counts, bad_counts, smooth=0.5):
    """Compute IV from per-bin Good/Bad counts with Laplace smoothing."""
    g = np.array(good_counts, dtype=float)
    b = np.array(bad_counts, dtype=float)
    # +0.5 on empty Good/Bad counts
    g = np.where(g == 0, smooth, g)
    b = np.where(b == 0, smooth, b)
    total_g = g.sum()
    total_b = b.sum()
    if total_g == 0 or total_b == 0:
        return 0.0
    dist_g = g / total_g
    dist_b = b / total_b
    iv = np.sum((dist_b - dist_g) * np.log(dist_b / dist_g))
    return float(iv)

def _safe_transform(ob, X):
    """Transform feature values with OptBinning, coercing mixed-type columns."""
    try:
        return ob.transform(X, metric="bins")
    except TypeError:
        return ob.transform(pd.to_numeric(X, errors="coerce"), metric="bins")

def get_bin_stats(ob, X, y_bin):
    """Get per-bin counts from a fitted OptBinning object applied to data."""
    bins = _safe_transform(ob, X)
    df = pd.DataFrame({"bin": pd.Series(bins).astype(str).values, "y": y_bin.values})
    stats = df.groupby("bin").agg(count=("y", "size"), bads=("y", "sum")).reset_index()
    stats["goods"] = stats["count"] - stats["bads"]
    stats["br"] = stats["bads"] / stats["count"]
    return stats

def get_full_dev_bin_stats(ob, X, y_bin):
    """Get full-DEV bin stats with bin labels for special/missing identification."""
    bins = _safe_transform(ob, X)
    df = pd.DataFrame({"bin": pd.Series(bins).astype(str).values, "y": y_bin.values})
    stats = df.groupby("bin").agg(count=("y", "size"), bads=("y", "sum")).reset_index()
    stats["goods"] = stats["count"] - stats["bads"]
    stats["br"] = stats["bads"] / stats["count"]
    # Identify special/missing bins
    stats["is_special"] = stats["bin"].apply(
        lambda b: str(b).lower() in ["missing", "special"] or str(b).startswith("Special")
    )
    return stats

def compute_woe_per_bin(good_counts, bad_counts, smooth=0.5):
    """Compute WoE per bin."""
    g = np.array(good_counts, dtype=float)
    b = np.array(bad_counts, dtype=float)
    g = np.where(g == 0, smooth, g)
    b = np.where(b == 0, smooth, b)
    total_g = g.sum()
    total_b = b.sum()
    if total_g == 0 or total_b == 0:
        return np.zeros(len(g))
    dist_g = g / total_g
    dist_b = b / total_b
    return np.log(dist_g / dist_b)

# Precompute full-DEV bin stats per feature
print("Computing full-DEV bin statistics...")
full_dev_stats = {}
for feat in l1_pass_feats:
    ob = binning_objects[feat]
    X_f = dev_gb[feat]
    full_dev_stats[feat] = get_full_dev_bin_stats(ob, X_f, y)

# Compute per-feature, per-period statistics
print("Computing per-period statistics...")
results = []

def process_feature(feat, ob, X_full, y_full, fd, period_labels, dev_gb, min_bin_ct):
    """Process one feature for L2 stability analysis."""
    fd_regular = fd[~fd["is_special"]].copy()

    # Head and tail bins (highest / lowest BR in full DEV, regular bins only)
    if len(fd_regular) >= 2:
        head_bin = fd_regular.loc[fd_regular["br"].idxmax(), "bin"]
        tail_bin = fd_regular.loc[fd_regular["br"].idxmin(), "bin"]
    else:
        head_bin = tail_bin = None

    row = {"feature": feat}
    rank_corrs = []
    has_ht_flip = False

    special_bin_names = fd[fd["is_special"]]["bin"].tolist()
    woe_signs = {sb: [] for sb in special_bin_names}

    for seg in period_labels:
        mask = dev_gb["segment"] == seg
        X_seg = X_full[mask]
        y_seg = y_full[mask]

        seg_stats = get_bin_stats(ob, X_seg, y_seg)
        iv_seg = compute_iv_from_counts(seg_stats["goods"].values, seg_stats["bads"].values)
        row[f"iv_{seg.lower()}"] = round(iv_seg, 6)

        # Rank correlation
        merged = fd_regular.merge(seg_stats, on="bin", how="inner", suffixes=("_full", "_seg"))
        merged["count_seg"] = pd.to_numeric(merged["count_seg"], errors="coerce")
        merged = merged[merged["count_seg"] >= min_bin_ct]
        if len(merged) >= 3:
            corr, _ = spearmanr(
                pd.to_numeric(merged["br_full"], errors="coerce"),
                pd.to_numeric(merged["br_seg"], errors="coerce")
            )
            rank_corrs.append(float(corr))
        else:
            rank_corrs.append(np.nan)

        # Head-tail flip
        if head_bin is not None and tail_bin is not None:
            h_row = seg_stats[seg_stats["bin"] == head_bin]
            l_row = seg_stats[seg_stats["bin"] == tail_bin]
            if len(h_row) > 0 and len(l_row) > 0:
                h_br = float(h_row["br"].values[0])
                l_br = float(l_row["br"].values[0])
                if h_br <= l_br:
                    has_ht_flip = True

        # WoE signs for special/missing bins
        for sb in special_bin_names:
            sb_in_seg = seg_stats[seg_stats["bin"] == sb]
            if len(sb_in_seg) > 0:
                seg_all = seg_stats.copy()
                woe_vals = compute_woe_per_bin(seg_all["goods"].values, seg_all["bads"].values)
                idx_sb = list(seg_all["bin"]).index(sb)
                if idx_sb < len(woe_vals):
                    woe_signs[sb].append(float(np.sign(woe_vals[idx_sb])))
                else:
                    woe_signs[sb].append(np.nan)
            else:
                woe_signs[sb].append(np.nan)

    valid_corrs = [rc for rc in rank_corrs if not np.isnan(rc)]
    row["rank_corr_min"] = min(valid_corrs) if valid_corrs else np.nan
    row["head_tail_flip"] = "Y" if has_ht_flip else "N"

    woe_flip = False
    for sb, signs in woe_signs.items():
        valid = [s for s in signs if not np.isnan(s)]
        if len(valid) >= 2 and len(set(valid)) > 1:
            woe_flip = True
            break
    row["woe_sign_flip"] = "Y" if woe_flip else "N"
    return row


for fi, feat in enumerate(l1_pass_feats):
    if (fi + 1) % 25 == 0:
        print(f"  [{fi+1}/{len(l1_pass_feats)}] {feat}")
    try:
        row = process_feature(
            feat, binning_objects[feat], dev_gb[feat], y,
            full_dev_stats[feat], period_labels, dev_gb, MIN_BIN_CT
        )
        results.append(row)
    except Exception as e:
        print(f"  ERROR on {feat}: {type(e).__name__}: {e}")
        row = {"feature": feat}
        for seg in period_labels:
            row[f"iv_{seg.lower()}"] = np.nan
        row["rank_corr_min"] = np.nan
        row["head_tail_flip"] = "N"
        row["woe_sign_flip"] = "N"
        results.append(row)

l2_df = pd.DataFrame(results)
print(f"\nComputed L2 stats for {len(l2_df)} features")
display(l2_df.head())

In [0]:
# ── L2 drop rules (spec 6.3.3) ─────────────────────────────────────

iv_cols = [c for c in l2_df.columns if c.startswith("iv_s")]

l2_df["L2_pass"] = "Y"
l2_df["L2_reason"] = pd.NA
l2_df["L2_flag"] = "N"
l2_df["L2_flag_reason"] = pd.NA

for idx in l2_df.index:
    reasons_drop = []
    reasons_flag = []

    # Rule 1: IV < seg_iv_min in >= drop_if_low_iv_segments_ge periods
    seg_ivs = l2_df.loc[idx, iv_cols].values.astype(float)
    n_low = (seg_ivs < SEG_IV_MIN).sum()
    if n_low >= DROP_LOW_GE:
        reasons_drop.append("L2_IV_UNSTABLE")

    # Rule 2: Head-tail reversal in any period -> drop
    if DROP_HT_FLIP and l2_df.loc[idx, "head_tail_flip"] == "Y":
        reasons_drop.append("HEAD_TAIL_FLIP")

    # Apply drops (first rule wins)
    if reasons_drop:
        l2_df.loc[idx, "L2_pass"] = "N"
        l2_df.loc[idx, "L2_reason"] = reasons_drop[0]
        continue

    # Flags (not dropped)
    # Flag 1: rank_corr < rank_corr_flag in any period
    rc_min = l2_df.loc[idx, "rank_corr_min"]
    if pd.notna(rc_min) and rc_min < RANK_CORR_FLAG:
        reasons_flag.append("RANK_CORR_LOW")

    # Flag 2: max(IV)/min(IV) > iv_ratio_flag (min > 0)
    iv_min_val = seg_ivs.min()
    iv_max_val = seg_ivs.max()
    if iv_min_val > 0 and (iv_max_val / iv_min_val) > IV_RATIO_FLAG:
        reasons_flag.append("IV_RATIO_HIGH")

    # Flag 3: WoE sign flip of missing/special bin
    if l2_df.loc[idx, "woe_sign_flip"] == "Y":
        reasons_flag.append("WOE_SIGN_FLIP")

    if reasons_flag:
        l2_df.loc[idx, "L2_flag"] = "Y"
        l2_df.loc[idx, "L2_flag_reason"] = ";".join(reasons_flag)

n_pass = (l2_df["L2_pass"] == "Y").sum()
n_drop = (l2_df["L2_pass"] == "N").sum()
n_flag = (l2_df["L2_flag"] == "Y").sum()
print(f"L2 results: {n_pass} pass ({n_flag} flagged), {n_drop} dropped")
print(f"\nDrop reasons:")
print(l2_df[l2_df["L2_pass"] == "N"]["L2_reason"].value_counts().to_string())
print(f"\nFlag reasons (among passed):")
flagged = l2_df[l2_df["L2_flag"] == "Y"]
if len(flagged) > 0:
    all_flag_reasons = []
    for r in flagged["L2_flag_reason"].dropna():
        all_flag_reasons.extend(r.split(";"))
    print(pd.Series(all_flag_reasons).value_counts().to_string())
else:
    print("  (none)")

In [0]:
# ── Update feature log ─────────────────────────────────────────────────────

l2_cols = ["feature"] + iv_cols + [
    "rank_corr_min", "head_tail_flip", "woe_sign_flip",
    "L2_flag", "L2_flag_reason", "L2_pass", "L2_reason"
]
l2_merge = l2_df[l2_cols].copy()

# Drop existing L2 columns from flog (idempotent re-run)
l2_only = [c for c in l2_merge.columns if c != "feature"]
flog_clean = flog.drop(columns=[c for c in l2_only if c in flog.columns])

flog_updated = flog_clean.merge(l2_merge, on="feature", how="left")

# Update dropped_at for L2 drops (only if not already dropped at L0/L1)
mask_l2_drop = (flog_updated["L2_pass"] == "N") & (flog_updated["dropped_at"].isna())
flog_updated.loc[mask_l2_drop, "dropped_at"] = "L2"
flog_updated.loc[mask_l2_drop, "drop_rule"] = flog_updated.loc[mask_l2_drop, "L2_reason"]

# Ensure no void columns
for c in flog_updated.columns:
    if flog_updated[c].isna().all():
        flog_updated[c] = flog_updated[c].astype(str)

# Save to Delta table
sdf = spark.createDataFrame(flog_updated)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL}  ({spark.table(FLOG_TBL).count()} rows)")

# Save CSV
csv_path = os.path.join(ARTIFACTS, "feature_log.csv")
flog_updated.to_csv(csv_path, index=False)
print(f"Saved CSV : {csv_path}")

# ── Report: reports/03_L2.xlsx ─────────────────────────────────────

tmp_xl = os.path.join(_TMP, "03_L2.xlsx")
with pd.ExcelWriter(tmp_xl, engine="openpyxl") as writer:
    meta = pd.DataFrame([{
        "module": "E6_L2", "run_time": datetime.now().isoformat(),
        "config_hash": common.config_hash(cfg),
        "funnel_config_hash": funnel_config_hash,
        "dev_rows_gb": len(dev_gb),
        "features_in": len(l2_df), "features_pass": int(n_pass),
        "features_drop": int(n_drop), "features_flagged": int(n_flag),
    }])
    meta.to_excel(writer, sheet_name="meta", index=False)

    # Period summary
    period_summary = []
    for seg in period_labels:
        sub = dev_gb[dev_gb["segment"] == seg]
        months = seg_map[seg_map == seg].index.sort_values()
        period_summary.append({
            "period": seg, "start": str(months[0]), "end": str(months[-1]),
            "n": len(sub), "bads": int(y[sub.index].sum()),
            "bad_rate": round(y[sub.index].mean(), 4)
        })
    pd.DataFrame(period_summary).to_excel(writer, sheet_name="periods", index=False)

    # Feature-level summary
    l2_df.sort_values(["L2_pass", "iv_s1"], ascending=[False, False]).to_excel(
        writer, sheet_name="feature_detail", index=False
    )

    # Per-feature period Bad Rate tables
    for feat in l1_pass_feats[:50]:  # first 50 to keep file manageable
        ob = binning_objects[feat]
        X_f = dev_gb[feat]
        sheet_rows = []
        for seg in period_labels:
            mask = dev_gb["segment"] == seg
            seg_stats = get_bin_stats(ob, X_f[mask], y[mask])
            seg_stats["period"] = seg
            sheet_rows.append(seg_stats)
        # Also add full DEV
        full_stats = get_bin_stats(ob, X_f, y)
        full_stats["period"] = "ALL_DEV"
        sheet_rows.append(full_stats)
        combined = pd.concat(sheet_rows, ignore_index=True)
        sheet_name = feat[:31]
        combined.to_excel(writer, sheet_name=sheet_name, index=False)

shutil.copyfile(tmp_xl, os.path.join(REPORTS, "03_L2.xlsx"))
print(f"Saved report: {REPORTS}/03_L2.xlsx")

In [0]:
# ── Acceptance Criteria (E6 — L2) ────────────────────────────────
print("=" * 60)
print("ACCEPTANCE CRITERIA (E6 — L2)")
print("=" * 60)
checks = []

# 1. Period Bad counts within ±25% of mean
period_bads = dev_gb.groupby("segment")[target_col].sum()
mean_b = period_bads.mean()
max_dev = ((period_bads - mean_b).abs() / mean_b).max()
checks.append((f"Period Bads within ±25% of mean (max dev={max_dev:.1%})", max_dev <= 0.25))
for seg in period_labels:
    b = period_bads[seg]
    pct = abs(b - mean_b) / mean_b
    print(f"  {seg}: {b} Bads ({pct:.1%} from mean {mean_b:.0f})")

# 2. Every L1-passed feature has L2 columns
l1_passed_in_log = flog_updated[(flog_updated["L1_pass"] == "Y")]
missing_l2 = l1_passed_in_log[l1_passed_in_log["L2_pass"].isna()]
checks.append((f"All L1-passed features have L2 columns ({len(missing_l2)} missing)", len(missing_l2) == 0))

# 3. No refit of binning (object hashes unchanged)
hash_changed = []
for feat in l1_pass_feats:
    pkl_path = os.path.join(BINNING_DIR, f"{feat}.pkl")
    with open(pkl_path, "rb") as f:
        current_hash = hashlib.sha256(f.read()).hexdigest()
    if current_hash != binning_hashes[feat]:
        hash_changed.append(feat)
checks.append((f"Binning objects unchanged ({len(hash_changed)} changed)", len(hash_changed) == 0))

# 4. feature_log.csv exported
checks.append(("feature_log.csv exported", os.path.exists(csv_path)))

# 5. Feature log table row count
tbl_ct = spark.table(FLOG_TBL).count()
checks.append((f"Feature log table rows = {tbl_ct}", tbl_ct == len(flog_updated)))

# 6. Dropped features have L2_reason
dropped_l2 = l2_df[l2_df["L2_pass"] == "N"]
if len(dropped_l2) > 0:
    has_reason = dropped_l2["L2_reason"].notna().all()
    checks.append(("Dropped features have L2_reason", has_reason))
else:
    checks.append(("Dropped features have L2_reason (none dropped)", True))

# 7. Report saved
checks.append(("03_L2.xlsx report saved", os.path.exists(os.path.join(REPORTS, "03_L2.xlsx"))))

all_pass = True
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
    if not ok:
        all_pass = False

# ── Run summary ─────────────────────────────────────────────────
summary = {
    "module": "E6_L2",
    "run_time": datetime.now().isoformat(),
    "elapsed_s": round(time.time() - _t0, 1),
    "config_hash": common.config_hash(cfg),
    "funnel_config_hash": funnel_config_hash,
    "dev_rows_gb": len(dev_gb),
    "features_in": len(l2_df),
    "features_pass": int(n_pass),
    "features_drop": int(n_drop),
    "features_flagged": int(n_flag),
    "drop_reasons": l2_df[l2_df["L2_pass"]=="N"]["L2_reason"].value_counts().to_dict(),
    "pass_fail": "PASS" if all_pass else "FAIL",
}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
with open(os.path.join(REPORTS, "run_summaries", "s03_L2.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary: {REPORTS}/run_summaries/s03_L2.json")

for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"

print("\n\u2705 All acceptance criteria passed.")